# Thiết Kế kiến trúc mạng VGG16

# import các thư viện cần thiết

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.transforms import transforms, functional
from sklearn.model_selection import train_test_split
from MLP import *

In [2]:
BaseMLP??

Init signature: BaseMLP()
Source:        
class BaseMLP(ABC):
    r"""
    abstract class - Lớp trừu tượng - Tạo bộ khung, mẫu cho class BASEMLP

    All subclass should overwrite metthod 'predict' to get prediction of model throught
    logits = self.Forward().
    Subclass also should overwrite metthod 'get_accuracy' to get accuracy from 2 parameter logits and y
    Subclass also should overwrite metthod 'compute_loss' to get loss_value from self.criterion
    """


    def __init__(self):
        self.Layers = []
        self.model = None
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.criterion = None
    
    @abstractmethod
    def predict(self,X):
        pass
    @abstractmethod
    def get_accuracy(self,logits,y):
        pass
    @abstractmethod
    def compute_loss(self,logits, y):
        pass
    
    def Add_layer(self,layers):
        if not isinstance(layers, list):
            raise TypeError("layers must be a list")
        self.Layer

In [3]:
class CNN(BaseMLP):
    def __init__(self):
        super().__init__()
    def predict(self, X):
        logits = self.forward(X)
        return torch.argmax(logits, dim = 1)
    def get_accuracy(self, logits, y):
        try:
            return torch.mean((torch.argmax(logits,dim =1) == torch.argmax(y,dim=1)).float())
        except:
            return torch.mean((torch.argmax(logits,dim =1) == y).float())
    def compute_loss(self, logits, y):
        return self.criterion(logits,y)

# Build VGG16 Standard

In [5]:
VGG16 = CNN()

# Block 1 
VGG16.Add_layer([
    nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding='same', stride = 1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding='same', stride =1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2)
])

# Block 2
VGG16.Add_layer([
    nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding = 'same', stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding = 'same', stride=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2)
])

# Block 3

VGG16.Add_layer([
    nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding = 'same', stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding = 'same', stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding = 'same', stride=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2)
])

# Block 4
VGG16.Add_layer([
    nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding='same', stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding='same', stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding='same', stride=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2)
])

# Block 5
VGG16.Add_layer([
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding='same', stride = 1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding='same', stride = 1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding='same', stride = 1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten()
])

# Block 6

VGG16.Add_layer([
    nn.Linear(in_features=7*7*512, out_features=4096),
    nn.ReLU(inplace=True),
    nn.Linear(in_features=4096, out_features=4096),
    nn.ReLU(inplace=True),
    nn.Linear(in_features=4096, out_features=1000)
])

In [6]:
summary(VGG16.model, input_size=(3,224,224), batch_size=32, device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [32, 64, 224, 224]           1,792
              ReLU-2         [32, 64, 224, 224]               0
            Conv2d-3         [32, 64, 224, 224]          36,928
              ReLU-4         [32, 64, 224, 224]               0
         MaxPool2d-5         [32, 64, 112, 112]               0
            Conv2d-6        [32, 128, 112, 112]          73,856
              ReLU-7        [32, 128, 112, 112]               0
            Conv2d-8        [32, 128, 112, 112]         147,584
              ReLU-9        [32, 128, 112, 112]               0
        MaxPool2d-10          [32, 128, 56, 56]               0
           Conv2d-11          [32, 256, 56, 56]         295,168
             ReLU-12          [32, 256, 56, 56]               0
           Conv2d-13          [32, 256, 56, 56]         590,080
             ReLU-14          [32, 256,